In [ ]:
import numpy as np
import cupy as cp
from cupyx.scipy import ndimage
from tanalysis import improcess
import tifffile as tiff
from skimage.measure import label, regionprops, regionprops_table
from skimage import exposure, morphology

In [ ]:
def LoG(sigma_x, sigma_y, sigma_z):
    n = 7
    z,y,x = cp.ogrid[-n//2:n//2+1, -n//2:n//2+1, -n//2:n//2+1]
    z_filter = cp.exp(-(z*z/(2*sigma_z**2)))
    y_filter = cp.exp(-(y*y/(2*sigma_y**2)))
    x_filter = cp.exp(-(x*x/(2*sigma_x**2)))
    final_filter = (1/(sigma_x*sigma_y*sigma_z*(2*np.pi)**(3/2)))*((1-x*x/(sigma_x**2))/(sigma_x**2)+(1-y*y/(sigma_y**2))/(sigma_y**2)+(1-z*z/(sigma_z**2))/(sigma_z**2))*(z_filter*x_filter*y_filter)
    return final_filter

In [ ]:
def LoG_convolve(img, sigmas_x, sigma_y, sigma_z):
    filter_log = LoG(sigmas_x, sigma_y, sigma_z)
    image = ndimage.convolve(img, filter_log)
    image = cp.square(image)
    return cp.asarray(image)

In [5]:
def fit_gaussian(sigma_x, sigma_y, sigma_z):
    n = 7
    z,y,x = cp.ogrid[-n//2:n//2+1, -n//2:n//2+1, -n//2:n//2+1]
    z_filter = cp.exp(-(z*z/(2*sigma_z**2)))
    y_filter = cp.exp(-(y*y/(2*sigma_y**2)))
    x_filter = cp.exp(-(x*x/(2*sigma_x**2)))
    final_filter = (x_filter*y_filter*z_filter)/(sigma_x*sigma_y*sigma_z*(2*np.pi)**(3/2))
    return final_filter

In [ ]:
def label_cells(image:np.ndarray, sigmas:list):
    '''
    '''
    Z,Y,X = image.shape
    og_img = np.double(image)
    img = cp.array(og_img)

    gaussian = fit_gaussian(*sigmas)
    img = ndimage.convolve(img, gaussian)
    th1 = cp.max(img)*0.1 #####################
    img_th1 = img>th1

    img = LoG_convolve(img, *sigmas)
    th2 = cp.max(img)*0.1 #####################
    img_th2 = img>th2

    img_th1 = cp.asnumpy(img_th1)
    img_th2 = cp.asnumpy(img_th2)

    img = img_th1 ^ img_th2
    img = morphology.dilation(img)
    img = morphology.erosion(img)
    img = morphology.remove_small_holes(img)
    img_r1 = morphology.remove_small_objects(img)

    img = cp.asarray(np.float16(img_r1))
    img = LoG_convolve(img, *sigmas)
    img = cp.asnumpy(img)
    th3 = np.max(img)*0.1
    img_th3 = img>th3
    img_r2 = img_r1 ^ img_th3
    img = morphology.erosion(img_r2)
    img = morphology.remove_small_objects(img)
    img = label(img)
    return img

In [50]:
dirname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.2.Test\EXP.HD6.1.2.Test.0\24h\Stitched\24h_chem20_thunder-0.tiff"
imgs, names, info = improcess.imread(dirname, False, False)

In [51]:
T,Z,Y,X = imgs[0].shape
result = np.empty(shape=(T,Z,Y,X), dtype=np.int8)
sigmas = (5,5,1)
t = 0
for img in imgs[0]:
    result[t] = label_cells(img, sigmas)
    t += 1

In [52]:
tiff.imwrite(r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.2.Test\EXP.HD6.1.2.Test.0\24h\Processed\24h_chem20.tiff", 
            np.uint8(result), 
            imagej=True,
            metadata={
                'axes':"TZYX"
            })

c:\Users\pcanaleta\Documents\python_codes\venvs\tanalysis\Lib\site-packages\tifffile\tifffile.py:3721: UserWarning: <tifffile.TiffWriter '24h_chem20.tiff'> truncating ImageJ file
  self._write_remaining_pages()
